In [1]:
import pandas as pd
import os
import glob
import numpy as np
from pathlib import Path
import socket
from zat.log_to_dataframe import LogToDataFrame

# ==========================================
# 1. CONFIGURATION (CHANGE HERE FOR ATTACKS)
# ==========================================
CATEGORY_DIR = "normal"
ATTACK_CAT = "Normal"
LABEL = 0

def list_files_with_extension(folder_path, extension):
    search_pattern = os.path.join(folder_path, f"*.{extension.lstrip('.')}")
    return glob.glob(search_pattern)

# Paths
BASE_DIR = os.getcwd()
INPUT_LOGS_DIR = os.path.join(BASE_DIR, "logs")
ZEEK_LOGS_DIR = os.path.join(INPUT_LOGS_DIR, "logs")
OUTPUT_FILE = os.path.join(BASE_DIR, f"merged_{CATEGORY_DIR}.csv") 

# Automatically fetch ALL Argus CSVs and Zeek logs
ARGUS_FILES = list_files_with_extension(INPUT_LOGS_DIR, "csv")
ZEEK_FILES = list_files_with_extension(ZEEK_LOGS_DIR, "log")

# ==========================================
# 2. LOAD & CLEAN ARGUS DATA
# ==========================================
print("Loading Argus flows...")
argus_dataframes = []

for file_path in ARGUS_FILES:
    print(f"Reading {file_path}...")
    df = pd.read_csv(file_path)
    argus_dataframes.append(df)

if not argus_dataframes:
    raise ValueError(f"No CSV files found in {INPUT_LOGS_DIR}")

argus_df = pd.concat(argus_dataframes, ignore_index=True)

argus_df.rename(columns={
    'SrcAddr': 'srcip', 'DstAddr': 'dstip', 'Sport': 'sport', 'Dport': 'dsport',
    'Proto': 'proto', 'State': 'state', 'Dur': 'dur', 'sTtl': 'sttl', 'dTtl': 'dttl',
    'SrcLoss': 'sloss', 'DstLoss': 'dloss', 'SrcPkts': 'Spkts', 'DstPkts': 'Dpkts',
    'SrcWin': 'swin', 'DstWin': 'dwin', 'SrcTCPBase': 'stcpb', 'DstTCPBase': 'dtcpb',
    'sMeanPktSz': 'smeansz', 'dMeanPktSz': 'dmeansz', 'SrcJitter': 'Sjit', 
    'DstJitter': 'Djit', 'SIntPkt': 'Sintpkt', 'DIntPkt': 'Dintpkt', 'TcpRtt': 'tcprtt',
    'SynAck': 'synack', 'AckDat': 'ackdat', 'SrcBytes': 'sbytes', 'DstBytes': 'dbytes',
    'SrcLoad': 'Sload', 'DstLoad': 'Dload', 'StartTime': 'Stime', 'LastTime': 'Ltime'
}, inplace=True)

argus_df['proto'] = argus_df['proto'].astype(str).str.lower()
argus_df['Stime'] = pd.to_numeric(argus_df['Stime'], errors='coerce')
argus_df = argus_df.dropna(subset=['Stime']) 
argus_df = argus_df.sort_values('Stime')

# ==========================================
# 3. LOAD & CLEAN ZEEK DATA (ROUTED LOGIC)
# ==========================================
print("Loading Zeek .log files...")
log_to_df = LogToDataFrame()

conn_dfs = []
http_dfs = []
ftp_dfs = []

# Dynamically route the files based on filename
for log_file in ZEEK_FILES:
    print(f"Reading {log_file}...")
    df = log_to_df.create_dataframe(log_file)
    if df is not None and not df.empty:
        if df.index.name == 'ts':
            df = df.reset_index()
            
        file_name = os.path.basename(log_file).lower()
        if 'conn' in file_name:
            conn_dfs.append(df)
        elif 'http' in file_name:
            http_dfs.append(df)
        elif 'ftp' in file_name:
            ftp_dfs.append(df)

if not conn_dfs:
    raise ValueError(f"No valid Zeek connection logs found in {ZEEK_LOGS_DIR}")

zeek_df = pd.concat(conn_dfs, ignore_index=True)

# Process HTTP Logs (Exact Method Extraction)
if http_dfs:
    http_df = pd.concat(http_dfs, ignore_index=True)
    if 'method' in http_df.columns:
        http_df['is_get_post'] = http_df['method'].astype(str).str.upper().isin(['GET', 'POST']).astype(int)
    else:
        http_df['is_get_post'] = 0
        
    http_agg = http_df.groupby('uid').agg({
        'trans_depth': 'max',
        'response_body_len': 'sum',
        'is_get_post': 'sum'
    }).reset_index()
    http_agg.rename(columns={'is_get_post': 'temp_http_mthd_count'}, inplace=True)
    zeek_df = pd.merge(zeek_df, http_agg, on='uid', how='left')

# Process FTP Logs (Exact Login & Command Extraction)
if ftp_dfs:
    ftp_df = pd.concat(ftp_dfs, ignore_index=True)
    
    # Count valid commands (ignore NaNs and Zeek's '-' placeholder)
    if 'command' in ftp_df.columns:
        ftp_df['has_cmd'] = (~ftp_df['command'].astype(str).isin(['-', 'nan', 'NaN'])).astype(int)
    else:
        ftp_df['has_cmd'] = 0

    # Determine if login occurred using Zeek's explicit user/password fields
    ftp_df['is_login'] = 0
    if 'user' in ftp_df.columns and 'password' in ftp_df.columns:
        has_user = ~ftp_df['user'].astype(str).isin(['-', 'nan', 'NaN'])
        has_pass = ~ftp_df['password'].astype(str).isin(['-', 'nan', 'NaN'])
        ftp_df['is_login'] = (has_user & has_pass).astype(int)
    elif 'command' in ftp_df.columns:
        # Fallback just in case standard fields are missing
        ftp_df['is_login'] = ftp_df['command'].astype(str).str.upper().isin(['USER', 'PASS', 'LOGIN']).astype(int)

    ftp_agg = ftp_df.groupby('uid').agg({
        'has_cmd': 'sum',
        'is_login': 'max'
    }).reset_index()
    ftp_agg.rename(columns={'has_cmd': 'temp_ftp_cmd_count', 'is_login': 'temp_is_ftp_login'}, inplace=True)
    zeek_df = pd.merge(zeek_df, ftp_agg, on='uid', how='left')

# Ensure all temporary and standard columns exist and fill NaNs safely
target_cols = ['trans_depth', 'response_body_len', 'temp_http_mthd_count', 'temp_ftp_cmd_count', 'temp_is_ftp_login']
for col in target_cols:
    if col not in zeek_df.columns:
        zeek_df[col] = 0
    zeek_df[col] = zeek_df[col].fillna(0)

# Standardize names
zeek_df.rename(columns={
    'id.orig_h': 'srcip', 'id.resp_h': 'dstip', 'id.orig_p': 'sport', 'id.resp_p': 'dsport',
    'proto': 'proto', 'response_body_len': 'res_bdy_len', 'ts': 'zeek_ts'
}, inplace=True)

zeek_df['proto'] = zeek_df['proto'].astype(str).str.lower()
zeek_df['zeek_ts_sec'] = zeek_df['zeek_ts'].astype('int64') / 10**9 
zeek_df['zeek_ts_sec'] = zeek_df['zeek_ts_sec'].astype(float)
zeek_df = zeek_df.sort_values('zeek_ts_sec')

# ==========================================
# 4. 5-TUPLE MERGE
# ==========================================
print("Merging Zeek and Argus on 5-Tuple...")

argus_df['Stime'] = argus_df['Stime'].astype(float)

def resolve_port(val):
    if pd.isna(val):
        return val
    try:
        return int(val)
    except (ValueError, TypeError):
        try:
            return socket.getservbyname(str(val).lower().strip())
        except OSError:
            return np.nan

for col in ['srcip', 'dstip', 'proto']:
    argus_df[col] = argus_df[col].astype(str)
    zeek_df[col] = zeek_df[col].astype(str)

for col in ['sport', 'dsport']:
    argus_df[col] = argus_df[col].apply(resolve_port).astype('Int64')
    zeek_df[col] = pd.to_numeric(zeek_df[col], errors='coerce').astype('Int64')

merged_df = pd.merge_asof(
    argus_df, 
    zeek_df[[
        'zeek_ts_sec', 'srcip', 'dstip', 'sport', 'dsport', 'proto', 'service', 
        'trans_depth', 'res_bdy_len', 'temp_http_mthd_count', 'temp_ftp_cmd_count', 'temp_is_ftp_login'
    ]], 
    left_on='Stime', 
    right_on='zeek_ts_sec',
    by=['srcip', 'dstip', 'sport', 'dsport', 'proto'],
    tolerance=2.0, 
    direction='nearest'
)

# ==========================================
# 5. UNSW-NB15 SCHEMA ALIGNMENT
# ==========================================
print("Aligning to UNSW-NB15 schema...")

merged_df['service'] = merged_df['service'].fillna('-')
merged_df['attack_cat'] = ATTACK_CAT
merged_df['Label'] = LABEL

merged_df['is_sm_ips_ports'] = np.where(
    (merged_df['srcip'] == merged_df['dstip']) & (merged_df['sport'] == merged_df['dsport']), 
    1, 0
)

unsw_columns = [
    'srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes',
    'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts',
    'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len',
    'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat',
    'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd',
    'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm',
    'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'Label'
]

# Add our temporary columns to the export list so Notebook 2 can see them
export_columns = unsw_columns + ['temp_http_mthd_count', 'temp_ftp_cmd_count', 'temp_is_ftp_login']

for col in export_columns:
    if col not in merged_df.columns:
        merged_df[col] = 0

final_df = merged_df[export_columns]

# ==========================================
# 6. EXPORT
# ==========================================
final_df.to_csv(OUTPUT_FILE, index=False)
print(f"Merge complete! Saved to {OUTPUT_FILE}")

Loading Argus flows...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\captura_normal_kube1.csv...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\captura_normal_kube2.csv...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\captura_normal_kube3.csv...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\captura_normal_kube4.csv...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\captura_normal_kube5.csv...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\captura_normal_kube6.csv...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\captura_normal_kube7.csv...


C:\Users\GabrielMoreira\AppData\Local\Temp\ipykernel_3996\3379182278.py:38: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\captura_normal_kube8.csv...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\captura_normal_kube9.csv...
Loading Zeek .log files...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\logs\conn.00_50_53-01_00_00.log...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\logs\conn.01_00_00-02_00_00.log...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\logs\conn.01_13_02-02_00_00.log...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\logs\conn.01_13_26-02_00_00.log...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\logs\logs\conn.02_00_00-03_00_00.log...
Reading C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\r

C:\Users\GabrielMoreira\AppData\Local\Temp\ipykernel_3996\3379182278.py:95: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  http_df = pd.concat(http_dfs, ignore_index=True)
C:\Users\GabrielMoreira\AppData\Local\Temp\ipykernel_3996\3379182278.py:111: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ftp_df = pd.concat(ftp_dfs, ignore_index=True)


Merging Zeek and Argus on 5-Tuple...
Aligning to UNSW-NB15 schema...
Merge complete! Saved to C:\Users\GabrielMoreira\log-anomaly-detection-ml\data\raw\ubuntu-server-logs\normal\merged_normal.csv
